# 02 · Phase 1 classification (v2)
Trains every run from its YAML (`configs/phase1_*.yaml`), each with W&B logging, then selects the kept model on the
**validation** split (highest macro-F1, tie → lower log-loss), calibrates it (temperature scaling) and analyses all
its test errors. Equivalent to `bash scripts/run_phase1_all.sh`.

## 0. Environment (Colab)
Clone the repository, install pinned requirements, set `FETALVISION_DATA`, mount Google Drive and unzip the Zenodo
archive (`FETAL_PLANES_ZENODO.zip`, record 3904280, CC BY 4.0) into `$FETALVISION_DATA/fetal_planes/raw/`.
All logic lives in the `fetalvision` package; this notebook only calls its CLIs.

In [ ]:
import os, subprocess, pathlib
if not pathlib.Path('FetalVision').exists():
    !git clone https://github.com/lamooooo1418/FetalVision.git
%cd FetalVision
!pip -q install -r requirements.txt -e .
os.environ['FETALVISION_DATA'] = '/content/data'
pathlib.Path('/content/data/fetal_planes/raw').mkdir(parents=True, exist_ok=True)
print('FETALVISION_DATA =', os.environ['FETALVISION_DATA'])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ZIP = '/content/drive/MyDrive/fetal planes/FETAL_PLANES_ZENODO.zip'   # adjust to where the Zenodo zip lives on your Drive
!test -f "$ZIP" && unzip -q -n "$ZIP" -d "$FETALVISION_DATA/fetal_planes/raw/extracted" || echo "zip not found: download https://zenodo.org/records/3904280"
!ls "$FETALVISION_DATA/fetal_planes/raw/extracted" | head

In [ ]:
# Weights & Biases: paste your API key when prompted (or leave WANDB_MODE=offline and `wandb sync` later)
import os
try:
    import wandb; wandb.login()
except Exception as e:
    os.environ['WANDB_MODE'] = 'offline'; print('W&B offline:', e)

## 1. Data (skip if notebook 01 was run in this session)

In [ ]:
!python -m fetalvision.data.prepare_planes --config configs/phase1_data.yaml --split-strategy group_shuffle --near-dup on
!python -m fetalvision.data.prepare_planes --config configs/phase1_data.yaml --split-strategy stratified_group --near-dup on

## 2. Training runs (one config per run; every run evaluates val + test once)

In [ ]:
!python -m fetalvision.classification.train --config configs/phase1_cnn_scratch.yaml

In [ ]:
!python -m fetalvision.classification.train --config configs/phase1_mnv3s_frozen.yaml

In [ ]:
!python -m fetalvision.classification.train --config configs/phase1_mnv3s_partial_ft.yaml

In [ ]:
!python -m fetalvision.classification.train --config configs/phase1_mnv3s_full_ft.yaml

In [ ]:
!python -m fetalvision.classification.train --config configs/phase1_mnv3s_full_ft_cw.yaml

In [ ]:
!python -m fetalvision.classification.train --config configs/phase1_mnv3s_partial_ft_cw.yaml

In [ ]:
!python -m fetalvision.classification.train --config configs/phase1_mnv3s_full_ft_cw_sgkf.yaml

## 3. Model selection on validation

In [ ]:
!python -m fetalvision.classification.select_model --results-root results
import json; sel = json.load(open('results/phase1/model_selection.json')); print(sel['rule']); SELECTED = sel['selected_run']; print('SELECTED:', SELECTED)

## 4. Calibration and error analysis of the selected run (and of the scratch baseline)

In [ ]:
for run in [SELECTED, 'p1v2-cnn-scratch']:
    !python -m fetalvision.classification.calibrate --run-name {run}
    !python -m fetalvision.classification.error_analysis --run-name {run}

## 5. Figures and numbers

In [ ]:
import json, pandas as pd
rows = []
for r in sel['ranked']:
    rows.append({k: r[k] for k in ['rank', 'run_name', 'val_macro_f1', 'val_log_loss', 'val_accuracy', 'test_accuracy', 'test_macro_f1', 'test_abdomen_recall', 'test_abdomen_thorax_confusions']})
pd.DataFrame(rows)

In [ ]:
import json, glob
from IPython.display import Image, display
for f in sorted(glob.glob('results/phase1/' + SELECTED + '/figures/*.png')):
    print(f); display(Image(filename=f, width=800))

In [ ]:
ea = json.load(open(f'results/phase1/{SELECTED}/error_analysis.json'))
print(json.dumps({k: ea[k] for k in ['consistency', 'errors_by_true_class', 'a_confidence', 'c_patients', 'e_second_choice']}, indent=1, default=str)[:4000])
cal = json.load(open(f'results/phase1/{SELECTED}/calibration.json')); print('T =', cal['temperature'], '| test ECE', cal['test']['ece_before'], '->', cal['test']['ece_after'])